# Задача 4 — НИР 2024
**БАЗАЕВ ИВАН KMBO-11-24** | **Вариант 3**

- Набор данных: HR Analytics: Job Change of Data Scientists
- Тип классификатора: SVM (метод опорных векторов)
- Классификация по: **Пол (gender)**

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')
print('Библиотеки загружены')

Библиотеки загружены


## Загрузка и подготовка данных

In [2]:
data = pd.read_csv('aug_train.csv')
print('Размер датасета:', data.shape)

# Удаляем строки без значения пола (целевой признак)
data1 = data.dropna(subset=['gender']).copy()
print('После удаления пропусков в gender:', data1.shape)

# Бинаризация целевого признака: Male = класс 0, Female/Other = класс 1
data1['gender_bin'] = np.where(data1['gender'] == 'Male', 0, 1)
print('\nРаспределение классов:')
print(data1['gender_bin'].value_counts())

Размер датасета: (19158, 14)
После удаления пропусков в gender: (14650, 14)

Распределение классов:
gender_bin
0    13221
1     1429
Name: count, dtype: int64


In [3]:
cat_cols = ['city', 'relevent_experience', 'enrolled_university',
            'education_level', 'major_discipline', 'experience',
            'company_size', 'company_type', 'last_new_job']
num_cols = ['city_development_index', 'training_hours']

X = data1.drop(['enrollee_id', 'gender', 'gender_bin', 'target'], axis=1)
Y = data1['gender_bin']

# Заполнение пропусков
X[cat_cols] = X[cat_cols].fillna('Unknown')
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# Тренировочная и тестовая выборки
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
print('Обучающая:', x_train.shape, '| Тестовая:', x_test.shape)

Обучающая: (10255, 11) | Тестовая: (4395, 11)


In [4]:
# Кодирование категориальных признаков
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_cat = ohe.fit_transform(x_train[cat_cols])
test_cat  = ohe.transform(x_test[cat_cols])

x_train_proc = np.hstack([x_train[num_cols].values, train_cat])
x_test_proc  = np.hstack([x_test[num_cols].values,  test_cat])

# Нормализация числовых признаков
scaler = StandardScaler()
x_train_proc[:, :len(num_cols)] = scaler.fit_transform(x_train_proc[:, :len(num_cols)])
x_test_proc[:,  :len(num_cols)] = scaler.transform(x_test_proc[:,  :len(num_cols)])

print('Обработанная обучающая выборка:', x_train_proc.shape)

Обработанная обучающая выборка: (10255, 189)


## 1. Классификатор SVM (метод опорных векторов)

In [5]:
# class_weight='balanced' — необходим из-за дисбаланса классов (~90% Male)
svm = SVC(class_weight='balanced')
parameters = {
    'kernel': ('linear', 'rbf'),
    'C': (0.1, 0.25, 0.5, 0.75, 1),
    'gamma': (1, 2, 3, 'scale', 'auto'),
    'decision_function_shape': ('ovo', 'ovr'),
    'shrinking': (True, False)
}
clf = GridSearchCV(svm, parameters, cv=5, n_jobs=-1, scoring='f1')
clf.fit(x_train_proc, y_train)
print('Лучшие параметры SVM:', clf.best_params_)

Лучшие параметры SVM: {'C': 1, 'decision_function_shape': 'ovo', 'gamma': 'auto', 'kernel': 'rbf', 'shrinking': True}


In [6]:
# Оцениваем точность SVM на тестовой выборке
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = clf.predict(x_test_proc)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print('=== SVM ===')
print(f'precision = {round(precision, 4)}')
print(f'recall    = {round(recall, 4)}')
print(f'f1        = {round(f1, 4)}')
precision_svm, recall_svm, f1_svm = precision, recall, f1

=== SVM ===
precision = 0.1554
recall    = 0.5218
f1        = 0.2395


## 2. Классификатор Random Forest (Случайный Лес)

Подбор гиперпараметров проводится итеративно:
1. **Базовый GridSearch** по `max_features`, `max_depth`, `criterion` (при фиксированном начальном числе деревьев).
2. **Двухэтапный перебор** (coarse-to-fine) по `min_samples_split` — сначала крупный шаг, затем мелкий вокруг найденного значения.
3. **Двухэтапный перебор** по `min_samples_leaf` — аналогично.
4. **Двухэтапный перебор по числу деревьев `n_estimators`** — сначала крупный шаг, затем мелкий вокруг найденного значения.

In [7]:
# Базовый GridSearch по гиперпараметрам случайного леса
N_TREES_INIT = 150  # начальное число деревьев (далее будет подобрано отдельно)

param_grid = {
    'max_features': ['sqrt', 'log2'],
    'max_depth':    list(range(1, 20)),
    'criterion':    ['gini', 'entropy']
}
rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=N_TREES_INIT),
    param_grid, cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf.fit(x_train_proc, y_train)
print('Лучшие параметры Random Forest:', rf.best_params_)

y_pred_rf = rf.predict(x_test_proc)
precision = precision_score(y_test, y_pred_rf)
recall    = recall_score(y_test, y_pred_rf)
f1        = f1_score(y_test, y_pred_rf)

print('\n=== Random Forest (базовый GridSearch) ===')
print(f'precision = {round(precision, 4)}')
print(f'recall    = {round(recall, 4)}')
print(f'f1        = {round(f1, 4)}')

# Зафиксируем лучшие значения из базового GridSearch для дальнейшего уточнения
best_max_features = rf.best_params_['max_features']
best_max_depth    = rf.best_params_['max_depth']
best_criterion    = rf.best_params_['criterion']

Лучшие параметры Random Forest: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'log2'}

=== Random Forest (базовый GridSearch) ===
precision = 0.171
recall    = 0.4891
f1        = 0.2534


### Итеративный (coarse-to-fine) перебор гиперпараметров

По схеме «сначала крупный шаг, затем мелкий» перебираем `min_samples_split`, `min_samples_leaf`, а затем **`n_estimators`** (число деревьев) — каждый параметр в два этапа.

In [8]:
# === Итеративный перебор: min_samples_split ===
# Первая итерация: крупный шаг
rf_mss1 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        n_estimators=N_TREES_INIT,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion
    ),
    {'min_samples_split': list(range(2, 51, 5))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_mss1.fit(x_train_proc, y_train)
best_mss_coarse = rf_mss1.best_params_['min_samples_split']
print(f'Первая итерация (min_samples_split, шаг 5): {best_mss_coarse}')

# Вторая итерация: мелкий шаг возле найденного значения
low  = max(2, best_mss_coarse - 5)
high = best_mss_coarse + 5
rf_mss2 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        n_estimators=N_TREES_INIT,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion
    ),
    {'min_samples_split': list(range(low, high + 1))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_mss2.fit(x_train_proc, y_train)
best_mss = rf_mss2.best_params_['min_samples_split']
print(f'Вторая итерация (min_samples_split, шаг 1): {best_mss}')

Первая итерация (min_samples_split, шаг 5): 12
Вторая итерация (min_samples_split, шаг 1): 9


In [9]:
# === Итеративный перебор: min_samples_leaf ===
# Первая итерация: крупный шаг
rf_msl1 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        n_estimators=N_TREES_INIT,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion,
        min_samples_split=best_mss
    ),
    {'min_samples_leaf': list(range(1, 31, 3))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_msl1.fit(x_train_proc, y_train)
best_msl_coarse = rf_msl1.best_params_['min_samples_leaf']
print(f'Первая итерация (min_samples_leaf, шаг 3): {best_msl_coarse}')

# Вторая итерация: мелкий шаг возле найденного значения
low  = max(1, best_msl_coarse - 3)
high = best_msl_coarse + 3
rf_msl2 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        n_estimators=N_TREES_INIT,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion,
        min_samples_split=best_mss
    ),
    {'min_samples_leaf': list(range(low, high + 1))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_msl2.fit(x_train_proc, y_train)
best_msl = rf_msl2.best_params_['min_samples_leaf']
print(f'Вторая итерация (min_samples_leaf, шаг 1): {best_msl}')

Первая итерация (min_samples_leaf, шаг 3): 1
Вторая итерация (min_samples_leaf, шаг 1): 1


In [10]:
# === Итеративный перебор: n_estimators (число деревьев) ===
# Первая итерация: крупный шаг 50
rf_n1 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion,
        min_samples_split=best_mss,
        min_samples_leaf=best_msl
    ),
    {'n_estimators': list(range(50, 501, 50))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_n1.fit(x_train_proc, y_train)
best_n_coarse = rf_n1.best_params_['n_estimators']
print(f'Первая итерация (n_estimators, шаг 50): {best_n_coarse}')

# Вторая итерация: мелкий шаг 10 возле найденного значения
low  = max(10, best_n_coarse - 50)
high = best_n_coarse + 50
rf_n2 = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced', random_state=42,
        max_features=best_max_features,
        max_depth=best_max_depth,
        criterion=best_criterion,
        min_samples_split=best_mss,
        min_samples_leaf=best_msl
    ),
    {'n_estimators': list(range(low, high + 1, 10))},
    cv=5, refit=True, n_jobs=-1, scoring='f1'
)
rf_n2.fit(x_train_proc, y_train)
best_n_estimators = rf_n2.best_params_['n_estimators']
print(f'Вторая итерация (n_estimators, шаг 10): {best_n_estimators}')

Первая итерация (n_estimators, шаг 50): 150
Вторая итерация (n_estimators, шаг 10): 140


In [11]:
# Финальная модель RF со всеми подобранными гиперпараметрами
rf_final = RandomForestClassifier(
    class_weight='balanced', random_state=42,
    n_estimators=best_n_estimators,
    max_features=best_max_features,
    max_depth=best_max_depth,
    criterion=best_criterion,
    min_samples_split=best_mss,
    min_samples_leaf=best_msl
)
rf_final.fit(x_train_proc, y_train)
y_pred_final = rf_final.predict(x_test_proc)

precision_rf = precision_score(y_test, y_pred_final)
recall_rf    = recall_score(y_test, y_pred_final)
f1_rf        = f1_score(y_test, y_pred_final)

print('Финальные параметры RF:')
print(f'  n_estimators       = {best_n_estimators}')
print(f'  max_features       = {best_max_features}')
print(f'  max_depth          = {best_max_depth}')
print(f'  criterion          = {best_criterion}')
print(f'  min_samples_split  = {best_mss}')
print(f'  min_samples_leaf   = {best_msl}')
print('\n=== Random Forest (после итеративного подбора) ===')
print(f'precision = {round(precision_rf, 4)}')
print(f'recall    = {round(recall_rf, 4)}')
print(f'f1        = {round(f1_rf, 4)}')

Финальные параметры RF:
  n_estimators       = 140
  max_features       = log2
  max_depth          = 5
  criterion          = gini
  min_samples_split  = 9
  min_samples_leaf   = 1

=== Random Forest (после итеративного подбора) ===
precision = 0.1644
recall    = 0.4847
f1        = 0.2456


## Итоговый вывод

In [12]:
print('=' * 60)
print(f'SVM:           precision={round(precision_svm,4)}, recall={round(recall_svm,4)}, f1={round(f1_svm,4)}')
print(f'Random Forest: precision={round(precision_rf,4)},  recall={round(recall_rf,4)},  f1={round(f1_rf,4)}')
print('=' * 60)
winner = 'SVM' if f1_svm >= f1_rf else 'Random Forest'
print(f'Лучший классификатор по F1: {winner}')

SVM:           precision=0.1554, recall=0.5218, f1=0.2395
Random Forest: precision=0.1644,  recall=0.4847,  f1=0.2456
Лучший классификатор по F1: Random Forest


In [13]:
# Важность признаков (Random Forest)
feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
importances = rf_final.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df.sort_values(by='Importance', ascending=False, inplace=True)
print(feat_df.head(15))

                                         Feature  Importance
138                  major_discipline_Humanities    0.066313
7                                  city_city_103    0.061150
0                         city_development_index    0.052983
142                     major_discipline_Unknown    0.036637
125   relevent_experience_No relevent experience    0.035829
164                               experience_>20    0.030568
163                                experience_<1    0.030281
124  relevent_experience_Has relevent experience    0.030170
1                                 training_hours    0.030137
131                  education_level_High School    0.025082
136                        major_discipline_Arts    0.022794
180                         company_type_Pvt Ltd    0.020217
177                             company_type_NGO    0.019468
86                                  city_city_50    0.018487
15                                 city_city_114    0.018273
